<a href="https://colab.research.google.com/github/tuankhoin/CO3057-Computer-Vision/blob/main/Week_8_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3057 - Digital Image Processing and Computer Vision

# Week 8 - Image Segmentation

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w8pts.JPG" height=500/>

---

## Segmentation

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/segmentation.png?raw=true" height=300/>


## Semantic / Instance Segmentation

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/instance_segmentation.png?raw=true" height=300/>

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/instance_segmentation2.png?raw=true" height=300/>

## Learning Objectives
- Implement clustering algorithms for segmentation and compare/contrast clustering methods
- Implement an algorithm for computing superpixels and explain their common applications
- Explain graph-based methods for image segmentation

---
# 1. Pixel-based Clustering
---

## Colour Clustering
- Cluster based on different colour distributions in the image
- How do we perform this clustering? K-Means, Mean Shift etc.

## 1.1 K-means
![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/k-means_6.png?raw=true)

($k = 6$ in picture)

- Setting $k$ number of clusters, initialize centroid guesses
- Iteratively finding the centroid in their given regions.
- Efficient

Parameters: number of clusters, initial guess
- often can lead to inaccuracies when the number of clusters chosen is too high or too low
- initial locations of the cluster centres greatly impacts the effectiveness of the algorithm to find the right cluster centroids

## 1.2 Gaussian Mixture Model ($k=6$)
![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/gaussian_mixture_model_k6.png?raw=true)

![](https://i.sstatic.net/Dua5N.png)

- a Bayesian approach to clustering
- accommodates the uncertainty of a pixel belonging to a particular pixel
- formed using Gaussian Distributions ⇒ Find the distribution that gives higest probability

This can be useful for much more complex situations where the **probability** of each pixel belonging to a cluster needs consideration.




## 1.3 Mean Shift

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/mean_shift_bandwidth7.png?raw=true)

- finding the _mode_ of a density
- iteratively shifts points in the plot towards the closest mode.

This results in a number of clusters and the ability to assign a sample to a cluster after fitting is complete. This also means that $k$ number of clusters do not need to be assigned to perform the algorithm

![](https://ai.stanford.edu/~syyeung/cvweb/gifs/mean%20shift.gif)


Steps:
- we take a region of our data points and calculate the centre of mass.
- We then shift the centre of our region to this centre of mass and calculate the next centre of mass
- We repeat this process until it converges with the peak of the PDF.

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/mean_shift_algorithm.png?raw=true)

## Mean Shift Segmentation
- Cluster in spatial+colour space; e.g.: ($x,y,R,G,B$) or ($x,;y,L,A,B$) coordinates

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/mean_shift_segmentation.png?raw=true)

## Mean Shift Parameters

Free parameters:
- kernel (commonly Gaussian)
- bandwidth (how wide the region is)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/mean_shift_parameters.png?raw=true)

## Important to Note:
- Pixel clustering only separates colour regions
  - In case two objects in different coordinates may have the same colour, we may need to include spatial coordinates of pixels as well.
- Putting more emphasis on coordinates produces a segmentation similar to the first image, relying only on colour produces the last image.
- For example, a zebra with black and white stripes will cluster each stripe into many clusters



---
# 2. Superpixels
---

## Superpixels
- **Oversegmentation** methods segment image into regions that are smaller than objects
	- Objects are separated from background
	- But objects are also separated into many parts
- Superpixels = groups of adjacent pixels with similar characteristics (e.g., colour or even texture)

## Superpixel Segmentation

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/superpixel_segmentation.png?raw=true)

## 2.1 SLIC Superpixel Algorithm
- Initialise cluster centres on non-edge pixels:
	- Initialise $k$ cluster centres $c_k = [x_k, y_k, I_k, b_k]$ by sampling the image in a regular grid
	- For each centre $c_k$, check an N x N neighbourhood around $c_k$ to find the pixel with lowest gradient (areas where the change is smooth and not drastic). Set $c_k$ to this pixel's $[x,y,l,a,b]$

- For each cluster centre $c_k$:
	- In a $2M$ x $2M$ square neighbourhood around $c_k$, measure pixel similarity to $c_k$
	- Assign pixels with similarity < threshold to cluster $k$
	- Compute new cluster centre $c_k$
- Repeat until average change in cluster centres (L1 distance) falls below a threshold
- Similarity measure:

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/similarity_measure.png?raw=true)

- Similarity metric does not guarantee that clusters will be connected pixels
- To enforce connectivity, pixels not connected to main cluster are re-assigned to closest adjacent cluster

## 2.2 Superpixel Methods

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/superpixel_methods.png?raw=true)

## 2.3 Superpixel Applications
- More compact representation for algorithms with high time complexity (600x800 pixels -> 200 superpixels)
- Common application: object segmentation
	- Oversegment image
	- Combine superpixels to find objects

## 2.4 Superpixel Merging
- Region Adjacency Graph (RAG)
	- Vertices = image regions (pixels or superpixels)
	- Edge weights = difference between regions
- To merge superpixels:
	- Identify edges below a threshold and re-label superpixels connected by these edges as one region
	- Or iteratively:
		- Find lowest-weight edge (greedy method), relabel connected superpixels as one region
		- Recompute RAG, repeat until a criterion is met (e.g., all edges above a threshold)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/superpixel_merging.png?raw=true)





---
# 3. Graph-Based Segmentation
---

## Images as Graphs
- Represent image as a graph $G = (V,E)$
	- Vertices = image regions (pixels or superpixels)
	- Edge weights = similarity between regions

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/images_as_graphs.png?raw=true)

## Graph Cuts
- Consider image as a fully-connected graph
- Partition graph into disjoint sets $A,B$ to maximise total edge weight = remove low-weight  edges between dissimilar regions
- Minimise value of cut:

$$
cut(A,B) = \sum_{u \in A , v \in B} w(u,v)
$$

$w(u,v)$ represents the weight of edge connecting $u$ and $v$
- This means we are removing the vertices that have the lowest sum of edge weights

## Graph Cuts
- Not ideal for image segmentation - tends to create small, isolated sets

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/graph_ccuts.png?raw=true)

## Normalised Cuts
- Instead of minimising cut value, minimise cut value as a fraction of total edge connections in entire graph (normalised cut)
- Normalised cut (Shi & Malik, 2000)

$$
Ncut(A,B) = \frac{cut(A,B)}{assoc(A,V)} + \frac{cut(A,B)}{assoc(B,V)}
$$

$$
Ncut(A,B) = \frac{\sum_{u \in A, v \in B}w(u,v)}{\sum_{u \in A, t \in V} w(u,t)} + \frac{\sum_{u \in A, v \in B}w(u, v)}{\sum_{v \in B, t \in V}w(v,t)}
$$

- What this means is that we will normalise the cut of one vertex with its weights associated with the entire graph (i.e., all the edges of the vertex in question), and do the same for the second vertex.
- This stops us from having small regions in our segmented images

## Normalised Cuts Results
- This is the result of normalised cut, which is starting to give us much more reasonably looking

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/normalised_cuts_results.png?raw=true)

## GrabCut
- Segments image pixels into just two classes: foreground (object) and background
- Uses colour clustering + graph cuts to find optimal classification of pixels into each class
- It forces the corners of the box to be background and then finds the pixels in the box that should be foreground

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/grabcut1.png?raw=true)

## GrabCut Algorithm
- Requires user to initialise algorithm with a bounding box

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/grabcut_algorithm.png?raw=true)

- For each class (foreground, background), represent distribution of pixel colour as a Gaussian mixture model (GMM)
- Represent image pixels as a graph (8-way connectivity)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/grabcut_algorithm2.png?raw=true)

- Denote the pixel graph as $G$ and the GMM as $\theta$
- $\alpha$ indicates label of each pixel (foreground or background)
- Iterate until convergence:
	- Find graph cut (label assignment) to minimise
		![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/graphcut_formula.png?raw=true)
	- Recompute GMM for new label assignment
 - Note: If we increase $\gamma$, we will get a smoother boundaries result but with pixels to be less likely to belong together being clustered. Decrease $\gamma$ and we get rougher boundaries but with pixels more likely to belong to the right cluster

## GraphCut Example

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/graphcut_example.png?raw=true)

## How to Get Labels?
So far we looked at segmentation but how do we do semantic segmentation?
- Unlabelled regions can be inputs to an object classification method
- Or, segmentation and classification can be done simulateneously

---
# Exercises & Code
---

Code for this lecture can be retrieved [here](https://colab.research.google.com/github/tuankhoin/COMP90086-Practical-Solutions/blob/master/2023/Week%2011.ipynb#scrollTo=t6gOWvZsKl38)

<img style="float: ;" src="https://raw.githubusercontent.com/saraao/COMP90086_image/main/penguins.jpg" width=600>

(1) Explain how a segmentation approach based on meanshift clustering of colour would operate on the image shown below. What regions/objects would you expect to be segmented in the final result and why?
> Answer: This method would cluster pixels with similar colours together, choosing a number of clusters based on bandwidth. The final result would:
- Probably segment the ocean well (almost constant colors)
- Not able to correctly segment the penguins
  - Black regions would be clustered with black patches on the background.
  - White regions would be clustered with white regions of the background.

(2) Explain how a superpixel segmentation combined with a region-merging approach would operate on the image shown below. What regions/objects would you expect to be segmented in the final result and why?
> Answer: This method would oversegment the image based on colour, then combine neighbouring regions with similar colours. The final result would:
- Probably segment the ocean well
- Not able to correctly segment the penguins
  - Black regions would be segmented
  - White regions would be merged with the background.


---
This part onwards is not examinable.

---
# 4. Modern Segmentation: Segment Anything & Beyond
---

## Motivation

In earlier sections, segmentation relied on:

* edge detection
* active contours (snakes)
* manual initialization and heuristics

These methods often struggle with:

* complex scenes
* varying lighting conditions
* large-scale datasets

Modern approaches aim to build **general-purpose segmentation models** that can work across **any image, any object, and minimal supervision**.

---

# 4.1 Segment Anything Model (SAM)

## Core Idea

The Segment Anything Model is a **foundation model for segmentation**.

Instead of training for one task, it is designed to:

> Segment **any object** in **any image** given a simple prompt.


## Inputs to SAM

![](https://d15shllkswkct0.cloudfront.net/wp-content/blogs.dir/1/files/2023/04/Screenshot-from-2023-04-04-08-23-51.png)

SAM takes two inputs:

1. Image $I$
2. Prompt $P$

Prompt can be:

* a point (foreground/background)
* a bounding box
* a rough mask


## Output

SAM outputs a segmentation mask:

$$
M(x,y) \in {0,1}
$$

representing the object region.

It can also output **multiple candidate masks** with confidence scores.

---

## 4.2 Model Architecture

![](https://learnopencv.com/wp-content/uploads/2023/04/segment-anything-model.png)

SAM consists of three main components:

### 1. Image Encoder

Transforms the image into a feature representation:

$$
F = E(I)
$$

Typically implemented using a **Vision Transformer (ViT)**.

### 2. Prompt Encoder

Encodes user input (points, boxes, masks):

$$
p = P(Prompt)
$$

### 3. Mask Decoder

Combines image features and prompt to predict masks:

$$
M = D(F, p)
$$


## Training Data

SAM is trained on an extremely large dataset:

* **SA-1B dataset**
* over **1 billion masks**
* diverse real-world images

This enables strong generalization across domains.


## Intuition

SAM behaves like a **universal segmentation tool**:

* click → segment object
* draw box → refine mask
* give hint → get full region

It shifts segmentation from:

> “train a model for a dataset”

to

> “use one model for everything”

---

## 4.3 Characteristics of SAM

Advantages

* general-purpose segmentation
* works on unseen objects
* interactive and flexible
* no task-specific training required

Limitations

* may produce multiple ambiguous masks
* struggles with fine boundaries in complex scenes
* does not inherently understand semantics (only regions)

---
# 5. Modern Trends in Segmentation
---

## 5.1 Foundation Models

Large models trained on massive datasets:

* SAM (segmentation)
* DINO / MAE (representation learning)

Key idea:

> Learn general visual features that transfer across tasks.

---

## 5.2 Prompt-Based Vision

Similar to NLP:

Instead of fixed inputs, models take **prompts**:

$$
Output = f(Image, Prompt)
$$

This enables:

* interactive segmentation
* flexible control
* multi-task behavior

---

## 5.3 Self-Supervised Learning

Reduce dependence on labeled data.

Models learn representations by solving proxy tasks:

* contrastive learning
* masked image modeling

---

## 5.4 Video & Temporal Segmentation

Extending segmentation to time:

$$
M(x,y,t)
$$

Applications:

* tracking objects
* video editing
* autonomous driving

---

## 5.5 3D Segmentation

Segmentation extended to 3D data:

* point clouds
* meshes
* neural fields

Closely related to modern scene representations such as NeRF.

---

## 5.6 Open-Vocabulary Segmentation

Models segment objects based on text input.

Example:

* "segment all chairs in the image"

This combines:

* vision models
* language models